# DeepPCB YOLOv8 Detection Baseline in Google Colab

This notebook is for the **DeepPCB COCO detection baseline only**.

- Model: `yolov8n.pt`
- Task: object detection
- Training style: fine-tuning / transfer learning from pretrained weights
- Dataset: `configs/deeppcb_coco_baseline.yaml`
- Scope: DeepPCB only, no PKU merge, no tiling, no segmentation, no severity yet


## Colab Setup Notes

Use one of these project access options:

1. Put the whole repo in Google Drive and mount Drive in Colab.
2. Clone your GitHub repo into `/content/`.
3. Upload the project folder manually if needed.

This notebook assumes **Google Drive** by default because it is the safest way to keep datasets, runs, and weights between sessions.

Important:

- The current DeepPCB COCO export has `trainval` and `test` splits.
- For this baseline workflow, `trainval` is used for training and `test` is reused as the validation/inference-check split.
- This keeps the workflow simple without changing the source dataset contents.


In [ ]:
!pip install -q ultralytics==8.4.14 opencv-python pyyaml matplotlib

import platform
import sys
import torch

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    !nvidia-smi
else:
    print('GPU not detected. In Colab, switch Runtime > Change runtime type > GPU before training.')


## Mount Google Drive and Open the Repo

Update `PROJECT_ROOT` if your repo folder has a different Drive location.


In [ ]:
from pathlib import Path

USE_DRIVE = True
PROJECT_ROOT = Path('/content/drive/MyDrive/PCB-Defect-Detector')  # change if needed

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

assert PROJECT_ROOT.exists(), f'Update PROJECT_ROOT to your repo folder: {PROJECT_ROOT}'

%cd $PROJECT_ROOT
print('Working directory:', PROJECT_ROOT)


## Prepare the DeepPCB YOLO Workspace

This runs the repo's DeepPCB preparation logic using the converted COCO config.
It does **not** start training yet.


In [ ]:
!python scripts/deeppcb_yolov8_colab_train.py --prepare-only
!cat configs/deeppcb_yolov8_baseline_data.yaml


## Baseline Training Settings

This is the first DeepPCB-only baseline run, so keep it short and practical.


In [ ]:
MODEL = 'yolov8n.pt'
EPOCHS = 10
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = '0'
FRACTION = 1.0
PATIENCE = 20
PROJECT_DIR = 'runs/deeppcb_baseline'
RUN_NAME = 'yolov8n_deeppcb_baseline_colab'

print({
    'model': MODEL,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'workers': WORKERS,
    'device': DEVICE,
    'fraction': FRACTION,
    'patience': PATIENCE,
    'project_dir': PROJECT_DIR,
    'run_name': RUN_NAME,
})


In [ ]:
import subprocess

train_cmd = [
    'python', 'scripts/deeppcb_yolov8_colab_train.py',
    '--model', MODEL,
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--workers', str(WORKERS),
    '--device', DEVICE,
    '--fraction', str(FRACTION),
    '--patience', str(PATIENCE),
    '--project', PROJECT_DIR,
    '--name', RUN_NAME,
]

print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, check=True)


## Check the Training Outputs

Ultralytics may place the run under `runs/detect/...`, so this cell checks both likely output locations.


In [ ]:
from pathlib import Path

candidate_run_dirs = [
    Path(PROJECT_DIR) / RUN_NAME,
    Path('runs/detect') / PROJECT_DIR / RUN_NAME,
]

RUN_DIR = None
for candidate in candidate_run_dirs:
    if candidate.exists():
        RUN_DIR = candidate
        break

assert RUN_DIR is not None, 'Training run folder not found yet.'

print('Run directory:', RUN_DIR)
print('Best weights:', RUN_DIR / 'weights' / 'best.pt')
print('Last weights:', RUN_DIR / 'weights' / 'last.pt')

results_csv = RUN_DIR / 'results.csv'
if results_csv.exists():
    import pandas as pd
    display(pd.read_csv(results_csv).tail())


## Quick Validation / Inference Check

This uses a few DeepPCB test images, which are also the validation split for this baseline workflow.


In [ ]:
from pathlib import Path
from IPython.display import Image, display
from ultralytics import YOLO

best_weights = RUN_DIR / 'weights' / 'best.pt'
assert best_weights.exists(), f'Missing best weights: {best_weights}'

val_dir = Path('data/coco_master/deeppcb_full/images/test')
sample_images = sorted(val_dir.rglob('*.jpg'))[:8]
assert sample_images, f'No DeepPCB test images found in {val_dir}'

pred_root = (Path.cwd() / 'data' / 'inspection_outputs').resolve()
pred_name = f'{RUN_NAME}_predictions'

model = YOLO(str(best_weights))
results = model.predict(
    source=[str(path) for path in sample_images],
    imgsz=IMGSZ,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(pred_root),
    name=pred_name,
    exist_ok=True,
    verbose=False,
)

pred_dir = pred_root / pred_name
print('Prediction directory:', pred_dir)
print('Predictions generated for', len(results), 'images')

for image_path in sorted(pred_dir.glob('*.jpg'))[:4]:
    display(Image(filename=str(image_path)))


## Notes

- This notebook stays DeepPCB-only and detection-only.
- The first DeepPCB run name is `yolov8n_deeppcb_baseline_colab`.
- Training artifacts will be saved under `runs/deeppcb_baseline` or `runs/detect/runs/deeppcb_baseline`, including `best.pt`, `last.pt`, and `results.csv`.
- Prediction images will be saved under `data/inspection_outputs/yolov8n_deeppcb_baseline_colab_predictions`.
- PKU, dataset merging, tiling, segmentation, severity, and synthetic augmentation stay for later steps.
- If you later run a longer DeepPCB baseline, change `RUN_NAME` so the first run stays preserved.
